# Diabetes — Model Training Pipeline
In this notebook, we train 6 machine learning models on metabolic indicators:
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Support Vector Machine (RBF)
5. K-Nearest Neighbors (KNN)
6. Gradient Boosting

In [1]:
import os
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

MODEL_DIR = "saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

## 1. Load Data and Split Features & Labels
Split into 80% training data and 20% testing data.

In [2]:
# Load cleaned diabetes dataset
df = pd.read_csv(os.path.join("processed_data", "cleaned_diabetes.csv"))

# Separate features (X) and target (y)
X = df.drop(columns=['Outcome'])
y = df['Outcome']

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

X_train shape: (1600, 8), y_train shape: (1600,)
X_test shape:  (400, 8), y_test shape:  (400,)


## 2. Feature Scaling (StandardScaler)
Standardize features and save `diabetes_scaler.pkl`.

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler
joblib.dump(scaler, os.path.join(MODEL_DIR, "diabetes_scaler.pkl"))
print("[OK] Saved scaler to saved_models/diabetes_scaler.pkl")

[OK] Saved scaler to saved_models/diabetes_scaler.pkl


## 3. Train and Save Models
- Logistic Regression, RBF SVM, and KNN use **scaled features**.
- Decision Tree, Random Forest, and Gradient Boosting use **raw features**.

In [4]:
models = {
    'diabetes_logistic_regression.pkl': (LogisticRegression(max_iter=1000, random_state=42), True),
    'diabetes_decision_tree.pkl': (DecisionTreeClassifier(max_depth=4, random_state=42), False),
    'diabetes_random_forest.pkl': (RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42), False),
    'diabetes_rbf_svm.pkl': (SVC(kernel='rbf', probability=True, random_state=42), True),
    'diabetes_knn.pkl': (KNeighborsClassifier(n_neighbors=7), True),
    'diabetes_gradient_boosting.pkl': (GradientBoostingClassifier(random_state=42), False)
}

for filename, (model, use_scaled) in models.items():
    X_train_data = X_train_scaled if use_scaled else X_train
    model.fit(X_train_data, y_train)
    save_path = os.path.join(MODEL_DIR, filename)
    joblib.dump(model, save_path)
    print(f"[OK] Trained & Saved: {filename}")

[OK] Trained & Saved: diabetes_logistic_regression.pkl
[OK] Trained & Saved: diabetes_decision_tree.pkl


[OK] Trained & Saved: diabetes_random_forest.pkl


[OK] Trained & Saved: diabetes_rbf_svm.pkl


[OK] Trained & Saved: diabetes_knn.pkl


[OK] Trained & Saved: diabetes_gradient_boosting.pkl


## 4. Verification
Check all files saved in `saved_models/`.

In [5]:
print("Diabetes model files:")
for f in sorted(os.listdir(MODEL_DIR)):
    if f.startswith("diabetes_"):
        print(" -", f)

Diabetes model files:
 - diabetes_decision_tree.pkl
 - diabetes_gradient_boosting.pkl
 - diabetes_knn.pkl
 - diabetes_logistic_regression.pkl
 - diabetes_random_forest.pkl
 - diabetes_rbf_svm.pkl
 - diabetes_scaler.pkl
